### Computer Vision (OpenCV & scikit-image) ###
* Edge Detection (Canny): Highlights the boundaries of objects by finding rapid changes in pixel intensity.

* Keypoint Detection (SIFT/ORB): Identifies unique points (corners or blobs) that are invariant to scale and rotation, useful for matching two images.

* Texture Analysis (LBP/HOG): Uses Local Binary Patterns (LBP) or Histogram of Oriented Gradients (HOG) to describe the "feel" or structural shape of an image.

* Histogram Comparison: Captures the distribution of intensities across the image


Convert random sample image to array

In [ ]:
import os
import random
from PIL import Image
import numpy as np

random.seed(42)

# image directory
img_files = r"../data/formatted/license_plate_detection/train/images"

# list files 
files = os.listdir(img_files)

# get random sample from files
test_file = os.path.join(img_files, random.sample(files, k=1)[0]) 

# open test file, convert to grayscale and convert to numpy array
img = Image.open(test_file)
gs_img = img.convert("L")
image_arr = np.asarray(gs_img)

**Edge Detection (Canny):** Highlights the boundaries of objects by finding rapid changes in pixel intensity.

In [ ]:
import matplotlib.pyplot as plt
from skimage import feature
%matplotlib inline

# determine edges based on different std dev values
# higher std dev means larger gradient, less edges

edges1 = feature.canny(image_arr)
edges2 = feature.canny(image_arr, sigma=3)

# display results
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(8, 3))

ax[0].imshow(image_arr, cmap='gray')
ax[0].set_title('noisy image')

ax[1].imshow(edges1, cmap='gray')
ax[1].set_title(r'Canny filter, $\sigma=1$')

ax[2].imshow(edges2, cmap='gray')
ax[2].set_title(r'Canny filter, $\sigma=3$')

plt.show()


**Keypoint Detection (SIFT/ORB):** Identifies unique points (corners or blobs) that are invariant to scale and rotation, useful for matching two images

In [ ]:

from skimage.feature import ORB, SIFT 
import matplotlib.pyplot as plt

# define orb extractor
orb_descriptor_extractor = ORB(n_keypoints=200)

# get ORB keypoints and descriptors from image
orb_descriptor_extractor.detect_and_extract(image_arr)
orb_keypoints = orb_descriptor_extractor.keypoints
orb_descriptors = orb_descriptor_extractor.descriptors

# define ORB coordinates for sample plot
x_coords_orb = orb_keypoints[:, 1]
y_coords_orb = orb_keypoints[:, 0]

# define SIFT extractor
sift_descriptor_extractor = SIFT()

# get SIFT keypoints and descriptors from image
sift_descriptor_extractor.detect_and_extract(image_arr)
sift_keypoints = sift_descriptor_extractor.keypoints
sift_descriptors = sift_descriptor_extractor.descriptors

# define SIFT coordinates for sample plot
x_coords_sift = sift_keypoints[:, 1]
y_coords_sift = sift_keypoints[:, 0]

# plot coordinates
fig, axes = plt.subplots(1,2)

axes[0].imshow(image_arr, cmap = "gray")
axes[0].set_title('ORB Keypoints')
axes[1].imshow(image_arr, cmap = "gray")
axes[1].set_title('SIFT Keypoints')
axes[0].scatter(x_coords_orb, y_coords_orb, label = "ORB keypoints", color = "red", s = 10, alpha = 0.25)
axes[1].scatter(x_coords_sift, y_coords_sift, label = "SIFT keypoints", color = "blue", s= 10,  alpha = 0.25)

plt.show()

**Texture Analysis (LBP):** Local Binary Patterns: Comparison of central pixel to its surrounding neighbors; if a neighbor's intensity is greater than or equal to the center, it's marked as 1, otherwise 0. These bits are then concatenated into a binary number and converted to decimal.

* P = Number of Neighbors
* R = Radius of Circle

In [ ]:
from skimage.feature import local_binary_pattern
P, R = 8,8
# Basic Syntax
lbp_image = local_binary_pattern(image_arr, P, R, method='default')

fig, axes = plt.subplots(1,2)

axes[1].imshow(lbp_image, cmap = "gray")
axes[1].set_title('LBP')
axes[0].imshow(image_arr, cmap = "gray")
axes[0].set_title('Original Image')

**Histogram of Oriented Gradients:** Image is divided into small cells and a 9-bin histogram of gradient directions is calculated for each, spanning 0 to 180.

In [ ]:
from skimage.feature import hog
from skimage import exposure
# Extract HOG features and a visualization image
fd, hog_image = hog(img, 
                    orientations=9, 
                    pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2), 
                    visualize=True, 
                    channel_axis=-1)

# Rescale histogram for better display
hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

fig, axes = plt.subplots(1,2)

axes[1].imshow(hog_image_rescaled, cmap = "gray")
axes[1].set_title('HOG')
axes[0].imshow(image_arr, cmap = "gray")
axes[0].set_title('Original Image')

plt.show()

**Histogram Comparison:** Histogram comparison using calc_hist and compare_hist from cv2 package. Compare hisograms to determine similarity or dissimilarity between two images

In [ ]:
import cv2
def compare_images(image_path1, image_path2):
    # 1. Load images
    img1 = cv2.imread(image_path1)
    img2 = cv2.imread(image_path2)
    
    if img1 is None or img2 is None:
        raise FileNotFoundError("One or both image paths are invalid.")

    # convert to HSV color space (better for color/lighting variations)
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # calculate histograms
    hist1 = cv2.calcHist([gray1], [0], None, [256], [0, 256])
    hist2 = cv2.calcHist([gray2], [0], None, [256], [0, 256])

    # normalize histograms (Crucial if images have different pixel counts)
    cv2.normalize(hist1, hist1, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    cv2.normalize(hist2, hist2, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)

    # compare using different metrics
    corr = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CORREL)
    chisq = cv2.compareHist(hist1, hist2, cv2.HISTCMP_CHISQR)
    intersect = cv2.compareHist(hist1, hist2, cv2.HISTCMP_INTERSECT)
    bhattacharyya = cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)

    # print results
    print(f"Correlation (Similarity): {corr:.4f}")
    print(f"Intersection (Similarity): {intersect:.4f}")
    print(f"Chi-Square (Distance/Dissimilarity): {chisq:.4f}")
    print(f"Bhattacharyya (Distance/Dissimilarity): {bhattacharyya:.4f}")

    
    fig, axes = plt.subplots(2,2)

    axes[0][0].imshow(img1)
    axes[0][0].set_title(image_path1.split("\\")[-1], fontsize = 6)

    axes[0][1].plot(hist1)
    axes[0][1].set_xlabel("Normalized Grayscale Value")
    axes[0][1].set_ylabel("Counts")
    
    axes[1][0].set_title(image_path2.split("\\")[-1],fontsize = 6)
    axes[1][0].imshow(img2)

    axes[1][1].plot(hist2)
    axes[1][1].set_xlabel("Normalized Grayscale Value")
    axes[1][1].set_ylabel("Counts")

    plt.tight_layout()
    plt.show()


tf1 = r"../data/formatted/license_plate_detection/test/images/lp_test_image097.jpg"
tf2 = r"../data/formatted/license_plate_detection/test/images/lp_test_image884.jpg"

compare_images(tf1, tf2)

**Compare Two Different Sample Images (More Dissimilar)**

In [ ]:
test_file1 = os.path.join(img_files, random.sample(files, k=3)[1]) 
test_file2 = os.path.join(img_files, random.sample(files, k=3)[2]) 

compare_images(test_file1, test_file2)

### EDA GOAL ###
* Use list (vector) of histogram bin counts as unique signature of each image
* Reduce vector to a smaller number of dimensions (2 or 3) using Principal Component Analysis
* Use unsupervised learning method K-Means or GMM to group images
* Visualize 

**Function to load image and return a normalized histogram of pixel values**

In [ ]:
def hist_arr(im_path):
    img = cv2.imread(im_path)
    
    if img is None:
        return FileNotFoundError("One or both image paths are invalid.")

    # convert to HSV color space (better for color/lighting variations)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # calculate histograms
    hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256])
    cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)

    return hist

test_hist1 = hist_arr(test_file1)

**Get random sampling of files in a dataset**

In [ ]:
# get random sample from files

files = random.sample(files, k=1000)
img_dict = {}
for file in files:
    tf_path = os.path.join(img_files, file) 

    hist_output = hist_arr(tf_path)
    img_dict[file] = hist_output.flatten()


**PCA Analysis**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# separate dictionary into image names and histogram bin counts
sorted_image_names = list(img_dict.keys())
flattened_histograms = list(img_dict.values())

# create 2D array of all histogram lists in list
histograms_matrix = np.array(flattened_histograms)

# normalize histogram values
X_norm = normalize(histograms_matrix, norm='l1', axis=1)

# PCA analysis (with 3 components)
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_norm)


**K-Means Clustering**

In [ ]:
from sklearn.cluster import KMeans

# define number of clusters
num_clusters = 5

# define K-Means model
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')

# run k-means on PCA array
cluster_labels = kmeans.fit_predict(X_pca)

# create dictionary with cluster groups keys and empty dictionary values
cluster_groups = {i: [] for i in range(num_clusters)}

# add images to dictionary based on their cluster label
for img_name, label in zip(sorted_image_names, cluster_labels):
    cluster_groups[label].append(img_name)

**Visualize cluster groups**

In [ ]:
 # images to show per row
samples_per_cluster = 5

# define figure and axes: rows = num_clusters, cols = samples_per_cluster
fig, axes = plt.subplots(num_clusters, samples_per_cluster, figsize=(12, 3 * num_clusters))

# define overall figure title
fig.suptitle("Samples per Cluster Group", fontsize=16, fontweight='bold')

# loop through cluster groupings
for cluster_id in range(num_clusters):
    
    # get images in a cluster group
    images_in_cluster = cluster_groups[cluster_id]

    # get minimum between samples selected and images in cluster
    num_to_sample = min(samples_per_cluster, len(images_in_cluster))

    # pick random sample from each cluster group
    sampled_images = random.sample(images_in_cluster, num_to_sample)
    
    # for each sample image in each cluster...
    for i in range(samples_per_cluster):
        
        # define an axis
        ax = axes[cluster_id, i]
        
        if i < len(sampled_images):
            # define image path
            img_name = sampled_images[i]
            img_path = os.path.join(img_files, img_name)
            
            try:
                # open image file
                img = Image.open(img_path)

                # show image on axis
                ax.imshow(img)

                # set title
                ax.set_title(f"{img_name}\n(Cluster {cluster_id})", fontsize=8)
            except FileNotFoundError:
                ax.text(0.5, 0.5, "File\nNot Found", ha='center', va='center', color='red', fontsize=9)
            except Exception as e:
                ax.text(0.5, 0.5, "Error\nLoading", ha='center', va='center', color='orange', fontsize=9)
        else:
            # Placeholder text if a cluster has fewer images than samples_per_cluster
            ax.text(0.5, 0.5, "No More\nImages", ha='center', va='center', color='gray', fontsize=9)
            
        ax.axis('off')

plt.tight_layout()
plt.show()

**Visualize PCA** 

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Enables 3D projection

# Initialize a 3D figure
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot using the 3 PCA dimensions
# 'cluster_labels' maps to your KMeans or clustering outputs
scatter = ax.scatter(
    X_pca[:, 0],  # Principal Component 1
    X_pca[:, 1],  # Principal Component 2
    X_pca[:, 2],  # Principal Component 3
    c=cluster_labels,  # Color points by cluster ID
    cmap='tab10',      # cmap to use
    s=50,              # Size of points
    alpha=0.8          # Transparency to see overlapping points
)

# Label the 3 dimensional axes
# Labels include the percentage of dataset variance explained by each axis
ax.set_xlabel(f'PC 1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC 2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_zlabel(f'PC 3 ({pca.explained_variance_ratio_[2]:.1%})')

# Add decorative and analytical elements
plt.title('3D PCA Space of Image Histograms', fontsize=14, fontweight='bold')
fig.colorbar(scatter, ax=ax, label='Cluster Assignment', pad=0.1)

plt.tight_layout()
plt.show()
